# Chapter 18 – Recursive versus iterative (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb)

The algorithms of the chapter, recursive and iterative, compared with **counts** (calls, iterations, comparisons, recursion depth) and with some measured times: **SumExplicitStack** (recursion simulated with a stack), recursive **MergeSort** and **MergeSortBottomUp**, the recursive fast power and **PowerIt**, **Kadane**, the three Fibonacci algorithms **FibRec**, **FibMemo** and **FibIt**, **GCDRec**/**GCDIt** and the recursive and iterative binary search (tail-recursion elimination). The counts are the same in every language and on every machine; the times (in ms) depend on the machine and on the language, only how they grow matters.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

Each program is written to a file by the `%%writefile` cell and then compiled and run with `g++ -std=c++17 -O2`: run the cells in order, from top to bottom. No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## 1. From recursion to an explicit stack: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$. The iterative version of the halving sum: the stack $P$ contains the pairs $(l,r)$ of the sub-arrays still to be summed and does the job of the call stack. The stack never contains more than $\Theta(\log_2 n)$ pairs.

On the array $a=[4,7,1,9]$ of the chapter we print the trace (with indices from 0: the pair $(1,4)$ of the chapter is here $(0,3)$; in each row the top of the stack is on the right) and we compare with `SumIterative`, `sumRecHalving` and `sumRecDecrease` of the code of the course, with the maximum depth of the recursion. Then we print the call stack of `sumRecDecrease(a, 3)` on $a=[4,7,1]$ (example of the chapter): the additions are performed only *after* the base case, while the stack is emptied, and at some point the active calls are $n+1$.

In [1]:
%%writefile explicit_stack.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int maxP = 0;   // maximum size of the stack P

typedef pair<int, int> Pair;

// SumExplicitStack(a, n): sum of a[0..n-1] with an explicit stack
long long SumExplicitStack(int *a, int n)
{
    long long s = 0;
    vector<Pair> P;                 // stack of the sub-arrays to be summed
    P.push_back({0, n - 1});
    maxP = max(maxP, (int)P.size());
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];           // base case
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            maxP = max(maxP, (int)P.size());
        }
    }
    return s;
}

string FmtP(const vector<Pair> &P)
{
    string s = "[";
    for (size_t i = 0; i < P.size(); i++)
    {
        if (i > 0) s += ",";
        s += "(" + to_string(P[i].first) + "," + to_string(P[i].second) + ")";
    }
    return s + "]";
}

long long SumExplicitStackTrace(int *a, int n)
{
    long long s = 0;
    vector<Pair> P = {{0, n - 1}};
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];
            printf("pop (%d,%d): base case, s=%lld => P=%s\n", l, r, s, FmtP(P).c_str());
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            printf("pop (%d,%d): mid=%d, push (%d,%d) and (%d,%d) => P=%s, s=%lld\n", l, r, mid, mid + 1, r, l, mid, FmtP(P).c_str(), s);
        }
    }
    return s;
}

// ---- the other sums of the course code, with the depth ----
int depth = 0;
int maxDepth = 0;

long long sumIt(int *a, int n)
{
    long long s = 0;
    for (int i = 0; i < n; i++)
    {
        s = s + a[i];
    }
    return s;
}

long long sumRecHalving(int *a, int l, int r)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long sumRecDecrease(int *a, int n)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (n == 0)
    {
        result = 0;
    }
    else
    {
        result = sumRecDecrease(a, n - 1) + a[n - 1];
    }
    depth--;
    return result;
}

// ---- the call stack of sumRecDecrease(a, 3) ----
int stepNo = 0;
int maxActive = 0;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

void Emit(const vector<string> &stack, const string &action)
{
    stepNo++;
    if ((int)stack.size() > maxActive) maxActive = stack.size();
    string st;
    for (size_t i = 0; i < stack.size(); i++)
    {
        if (i > 0) st += " ";
        st += stack[i];
    }
    printf("%2d | %s | %s\n", stepNo, st.c_str(), action.c_str());
}

long long SumRecDecreaseTrace(int *a, int n, vector<string> &stack)
{
    stack.push_back("s(" + to_string(n) + ")");
    if (n == 0)
    {
        Emit(stack, "n=0: return 0");
        stack.pop_back();
        return 0;
    }
    Emit(stack, "n=" + to_string(n) + " != 0: call s(" + to_string(n - 1) + "), then add a[" + to_string(n - 1) + "]");
    long long r = SumRecDecreaseTrace(a, n - 1, stack);
    Emit(stack, "return " + to_string(r) + "+a[" + to_string(n - 1) + "]=" + to_string(r + a[n - 1]));
    stack.pop_back();
    return r + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {4, 7, 1, 9};
    int n = 4;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("SumExplicitStack(a, 4):\n");
    long long res = SumExplicitStackTrace(a, n);
    maxP = 0;
    SumExplicitStack(a, n);
    printf("result %lld, maximum size of P = %d\n", res, maxP);
    printf("\n");
    printf("sumIt                = %lld\n", sumIt(a, n));
    maxDepth = 0;
    long long r1 = sumRecHalving(a, 0, n - 1);
    printf("sumRecHalving        = %lld   maximum depth = %d\n", r1, maxDepth);
    maxDepth = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease       = %lld   maximum depth = %d\n", r2, maxDepth);

    int b[] = {4, 7, 1};
    printf("\n");
    printf("a = %s   sumRecDecrease(a, 3)\n", Fmt(b, 3).c_str());
    printf(" # | stack of the active calls | action\n");
    vector<string> stack;
    res = SumRecDecreaseTrace(b, 3, stack);
    printf("result %lld, active calls at most = %d = n+1\n", res, maxActive);
    return 0;
}
#endif

Writing explicit_stack.cpp


In [2]:
!g++ -std=c++17 -O2 -o explicit_stack explicit_stack.cpp && ./explicit_stack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) and (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) and (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): base case, s=4 => P=[(2,3),(1,1)]
pop (1,1): base case, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) and (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): base case, s=12 => P=[(3,3)]
pop (3,3): base case, s=21 => P=[]
result 21, maximum size of P = 3

sumIt                = 21
sumRecHalving        = 21   maximum depth = 3
sumRecDecrease       = 21   maximum depth = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | stack of the active calls | action
 1 | s(3) | n=3 != 0: call s(2), then add a[2]
 2 | s(3) s(2) | n=2 != 0: call s(1), then add a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: call s(0), then add a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
result 12, active calls at most = 4 = n+1


**Try it yourself.** Change the array: the explicit stack never contains more than $\lceil\log_2 n\rceil+1$ pairs.

In [3]:
%%writefile prova_explicit_stack.cpp
#define NOMAIN
#include "explicit_stack.cpp"

int main()
{
    // Change the array: the explicit stack never contains more than ceil(log2 n) + 1 pairs.
    vector<int> c(100);
    for (int i = 0; i < 100; i++) c[i] = i + 1;
    maxP = 0;
    long long s = SumExplicitStack(c.data(), 100);
    printf("%lld %d\n", s, maxP);
    return 0;
}

Writing prova_explicit_stack.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_explicit_stack prova_explicit_stack.cpp && ./prova_explicit_stack

5050 8


## 2. Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [5]:
%%writefile bottom_up.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // comparisons between elements

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r) of chapter 17
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;
    int n2 = r - m;
    vector<int> L(n1), R(n2);
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

void MergeSort(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSort(a, l, m);
        MergeSort(a, m + 1, r);
        Merge(a, l, m, r);
    }
}

// MergeSortBottomUp(a, n): sorts a[0..n-1] without recursion
void MergeSortBottomUp(int *a, int n)
{
    int width = 1;
    while (width < n)
    {
        int l = 0;
        while (l + width < n)                   // pseudo-code: l + width <= n
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            l = l + 2 * width;
        }
        width = 2 * width;
    }
}

void MergeSortBottomUpTrace(int *a, int n)
{
    int width = 1;
    int passes = 0;
    while (width < n)
    {
        int l = 0;
        string parts;
        while (l + width < n)
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            if (!parts.empty()) parts += ", ";
            parts += "a[" + to_string(l) + ".." + to_string(m) + "] with a[" + to_string(m + 1) + ".." + to_string(r) + "]";
            l = l + 2 * width;
        }
        passes++;
        printf("width=%d: merge %s => %s\n", width, parts.c_str(), Fmt(a, n).c_str());
        width = 2 * width;
    }
    printf("passes = %d\n", passes);
}

// InsertionSort(a, n) (chapter 3), with the count of comparisons
void InsertionSort(int *a, int n)
{
    for (int j = 1; j < n; j++)
    {
        int key = a[j];
        int i = j - 1;
        while (i >= 0)
        {
            comparisons++;
            if (a[i] > key)
            {
                a[i + 1] = a[i];
                i = i - 1;
            }
            else
            {
                break;
            }
        }
        a[i + 1] = key;
    }
}

// Linear congruential generator: same numbers in Python, C++ and Java
vector<int> RandomArray(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = 1 + (int)(x % 1000000ULL);
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {5, 2, 4, 6, 1, 3};
    printf("a = %s   n = %d\n", Fmt(a, 6).c_str(), 6);
    MergeSortBottomUpTrace(a, 6);

    printf("\n");
    printf("%6s %14s %10s %9s  comparisons\n", "n", "InsertionSort", "MergeSort", "BottomUp");
    vector<array<double, 4>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> base = RandomArray(n);
        vector<int> x1 = base, x2 = base, x3 = base;
        comparisons = 0;
        double t0 = NowMs(); InsertionSort(x1.data(), n); double t1 = NowMs() - t0;
        long long c1 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSort(x2.data(), 0, n - 1); double t2 = NowMs() - t0;
        long long c2 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSortBottomUp(x3.data(), n); double t3 = NowMs() - t0;
        long long c3 = comparisons;
        vector<int> ord = base;
        sort(ord.begin(), ord.end());
        const char *ok = (x1 == x2 && x2 == x3 && x3 == ord) ? "OK" : "ERROR";
        printf("%6d %14lld %10lld %9lld  %s\n", n, c1, c2, c3, ok);
        times.push_back({(double)n, t1, t2, t3});
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (auto &t : times)
    {
        printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms\n", (int)t[0], t[1], t[2], t[3]);
    }
    return 0;
}
#endif

Writing bottom_up.cpp


In [6]:
!g++ -std=c++17 -O2 -o bottom_up bottom_up.cpp && ./bottom_up

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort      0.077 ms   MergeSort    0.062 ms   MergeSortBottomUp    0.061 ms
n=2000   InsertionSort      0.315 ms   MergeSort    0.121 ms   MergeSortBottomUp    0.112 ms
n=4000   InsertionSort      1.200 ms   MergeSort    0.259 ms   MergeSortBottomUp    0.239 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [7]:
%%writefile prova_bottom_up.cpp
#define NOMAIN
#include "bottom_up.cpp"

int main()
{
    // Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
    int n = 8000;
    vector<int> base = RandomArray(n);
    vector<int> x1 = base, x2 = base;
    comparisons = 0;
    InsertionSort(x1.data(), n);
    long long c1 = comparisons;
    comparisons = 0;
    MergeSortBottomUp(x2.data(), n);
    printf("%lld %lld %s\n", c1, comparisons, x1 == x2 ? "True" : "False");
    return 0;
}

Writing prova_bottom_up.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_bottom_up prova_bottom_up.cpp && ./prova_bottom_up

16115652 93756 True


## 3. Fast power: recursive and PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — recursive: $x^n=(x^{\lfloor n/2\rfloor})^2$ if $n$ is even, $(x^{\lfloor n/2\rfloor})^2\cdot x$ if it is odd. $\Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor+2$ calls (depth $\Theta(\log_2 n)$). The result of the recursive call is stored in $y$ and used twice.
- **PowerIt**$(x,n)$ — *Input*: a number $x$ and a natural number $n$. *Output*: $x^n$. It reads the bits of $n$ from the least significant one: $\lfloor\log_2 n\rfloor+1$ iterations, extra memory $\Theta(1)$; $p\cdot y^k=x^n$ holds at every iteration.

With $x=2$ and $n=13=(1101)_2$ we print the trace of the chapter: $p$ is multiplied by $x$, $x^4$ and $x^8$ (the bits of 13 equal to 1) and $p=x^{13}$ after $4$ iterations. Then the table: same result, $\lfloor\log_2 n\rfloor+1$ iterations against $\lfloor\log_2 n\rfloor+2$ calls, against the $n$ multiplications of the naive loop.

In [9]:
%%writefile power.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int powCalls = 0;   // calls of FastPower
int powIters = 0;   // iterations of PowerIt

// FastPower(x, n): x^n by recursion on the halving of n
long long FastPower(long long x, int n)
{
    powCalls++;
    if (n == 0)
    {
        return 1;                   // base case
    }
    long long y = FastPower(x, n / 2);  // one recursive call
    if (n % 2 == 0)
    {
        return y * y;               // n even
    }
    return x * y * y;               // n odd
}

// PowerIt(x, n): x^n with square-and-multiply
long long PowerIt(long long x, int n)
{
    long long p = 1;
    long long y = x;
    int k = n;
    while (k > 0)
    {
        powIters++;
        if (k % 2 == 1)
        {
            p = p * y;
        }
        y = y * y;
        k = k / 2;
    }
    return p;
}

void PowerItTrace(long long x, int n)
{
    string bits;
    for (int t = n; t > 0; t /= 2) bits = char('0' + t % 2) + bits;
    printf("PowerIt(%lld, %d): %d = (%s)_2\n", x, n, n, bits.c_str());
    long long p = 1, y = x;
    int k = n, it = 0;
    while (k > 0)
    {
        it++;
        const char *par = k % 2 == 1 ? "odd" : "even";
        if (k % 2 == 1)
        {
            p = p * y;
        }
        y = y * y;
        printf("%d. k=%d %s: p=%lld, y=%lld, k=%d\n", it, k, par, p, y, k / 2);
        k = k / 2;
    }
    printf("iterations = %d, result p = %lld\n", it, p);
}

#ifndef NOMAIN
int main()
{
    PowerItTrace(2, 13);
    printf("\n");
    printf("%3s %12s %8s %10s %6s   (iterations, calls, multiplications)\n", "n", "2^n", "PowerIt", "FastPower", "naive");
    int ns[] = {0, 1, 2, 5, 13, 30};
    for (int n : ns)
    {
        powIters = 0;
        long long r1 = PowerIt(2, n);
        powCalls = 0;
        long long r2 = FastPower(2, n);
        printf("%3d %12lld %8d %10d %6d   %s\n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1LL << n)) ? "OK" : "ERROR");
    }
    return 0;
}
#endif

Writing power.cpp


In [10]:
!g++ -std=c++17 -O2 -o power power.cpp && ./power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 odd: p=2, y=4, k=6
2. k=6 even: p=2, y=16, k=3
3. k=3 odd: p=32, y=256, k=1
4. k=1 odd: p=8192, y=65536, k=0
iterations = 4, result p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterations, calls, multiplications)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Try it yourself.** Try another base and another exponent (in C++ and Java the result must fit in 64 bits).

In [11]:
%%writefile prova_power.cpp
#define NOMAIN
#include "power.cpp"

int main()
{
    // Try another base and another exponent (beware: in C++ and Java the result must fit in 64 bits).
    powIters = 0;
    powCalls = 0;
    long long r1 = PowerIt(3, 25);
    long long r2 = FastPower(3, 25);
    printf("%lld %d %lld %d\n", r1, powIters, r2, powCalls);
    return 0;
}

Writing prova_power.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_power prova_power.cpp && ./prova_power

847288609443 5 847288609443 6


## 4. Kadane: a better iterative algorithm

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the maximum sum of a non-empty subarray $a[l..r]$. It is the compact version of the chapter: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. The loop makes $n-1$ iterations: $\Theta(n)$ steps and $\Theta(1)$ memory, against the $n(n+1)/2$ pairs $(l,r)$ of brute force ($\Theta(n^2)$) and against the recursive divide and conquer ($\Theta(n\log_2 n)$): a good iterative algorithm can beat recursion.

On the example of the price changes of chapter 17 the result is $43$. Then, on pseudo-random arrays with values between $-50$ and $50$ (same generator in all languages, seed $n$), the iterations of `Kadane`, the pairs examined by brute force and the check that the two algorithms find the same maximum sum; finally the measured times.

In [13]:
%%writefile kadane_compact.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Kadane(a, n): maximum sum of a non-empty subarray of a[0..n-1]
int Kadane(int *a, int n)
{
    int best = a[0];
    int cur = a[0];                 // maximum sum of a subarray ending at i
    int i = 1;
    while (i < n)
    {
        cur = max(a[i], cur + a[i]);
        best = max(best, cur);
        i = i + 1;
    }
    return best;
}

// Brute force: maximum sum and number of pairs (l, r) examined
int BruteForceBest(int *a, int n, long long &pairs)
{
    int best = a[0];
    pairs = 0;
    for (int l = 0; l < n; l++)
    {
        int s = 0;
        for (int r = l; r < n; r++)
        {
            s = s + a[r];
            pairs++;
            if (s > best) best = s;
        }
    }
    return best;
}

// Integers between -50 and 50 (linear congruential generator)
vector<int> RandomValues(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = (int)(x % 101ULL) - 50;
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    printf("a = %s\n", Fmt(a, 16).c_str());
    printf("Kadane(a, 16) = %d\n", Kadane(a, 16));
    int b[] = {-3, -1, -2, -5};
    printf("a = %s  Kadane(a, 4) = %d\n", Fmt(b, 4).c_str(), Kadane(b, 4));
    printf("\n");
    printf("%6s %7s %11s %11s %9s  same sum?\n", "n", "Kadane", "BruteForce", "iterations", "pairs");
    vector<array<double, 3>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> v = RandomValues(n, n);
        long long pairs;
        double t0 = NowMs(); int k = Kadane(v.data(), n); double t1 = NowMs() - t0;
        t0 = NowMs(); int bf = BruteForceBest(v.data(), n, pairs); double t2 = NowMs() - t0;
        printf("%6d %7d %11d %11d %9lld  %s\n", n, k, bf, n - 1, pairs, k == bf ? "OK" : "ERROR");
        times.push_back({(double)n, t1, t2});
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (auto &t : times)
    {
        printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms\n", (int)t[0], t[1], t[2]);
    }
    return 0;
}
#endif

Writing kadane_compact.cpp


In [14]:
!g++ -std=c++17 -O2 -o kadane_compact kadane_compact.cpp && ./kadane_compact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  same sum?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Measured times (they depend on the machine):
n=1000   Kadane    0.0008 ms   BruteForce      0.209 ms
n=2000   Kadane    0.0014 ms   BruteForce      0.841 ms
n=4000   Kadane    0.0029 ms   BruteForce      3.390 ms


**Try it yourself.** Change $n$: the pairs of brute force grow as $n^2$, the iterations of Kadane as $n$.

In [15]:
%%writefile prova_kadane_compact.cpp
#define NOMAIN
#include "kadane_compact.cpp"

int main()
{
    // Try n = 8000: the pairs of brute force are n(n+1)/2 = 32 004 000, the iterations of Kadane 7999.
    int n = 8000;
    vector<int> v = RandomValues(n);
    long long pairs;
    int bf = BruteForceBest(v.data(), n, pairs);
    printf("%d (%d, %lld)\n", Kadane(v.data(), n), bf, pairs);
    return 0;
}

Writing prova_kadane_compact.cpp


In [16]:
!g++ -std=c++17 -O2 -o prova_kadane_compact prova_kadane_compact.cpp && ./prova_kadane_compact

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [17]:
%%writefile fibonacci.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long fibCalls = 0;   // calls executed

// FibRec(n)
long long FibRec(int n)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    return FibRec(n - 1) + FibRec(n - 2);
}

// FibMemo(n, memo): memo[k] = -1 means "undefined"
long long FibMemo(int n, long long *memo)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    if (memo[n] == -1)
    {
        memo[n] = FibMemo(n - 1, memo) + FibMemo(n - 2, memo);
    }
    return memo[n];
}

// FibIt(n)
long long FibIt(int n)
{
    if (n <= 1)
    {
        return n;
    }
    long long prev = 0;             // F(i-2)
    long long curr = 1;             // F(i-1)
    int i = 2;
    while (i <= n)
    {
        long long next = prev + curr;
        prev = curr;
        curr = next;
        i = i + 1;
    }
    return curr;
}

// Like FibRec, but counts the calls for each value of the parameter
long long FibRecHist(int n, int *counts)
{
    counts[n]++;
    if (n <= 1)
    {
        return n;
    }
    return FibRecHist(n - 1, counts) + FibRecHist(n - 2, counts);
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int counts[6] = {0, 0, 0, 0, 0, 0};
    long long f5 = FibRecHist(5, counts);
    printf("FibRec(5) = %lld\n", f5);
    string line;
    int total = 0;
    for (int k = 5; k >= 0; k--)
    {
        if (k < 5) line += "  ";
        line += to_string(k) + ":" + to_string(counts[k]);
        total += counts[k];
    }
    printf("calls for each value of the parameter: %s   total %d\n", line.c_str(), total);
    printf("\n");
    printf("%3s %8s %9s %10s %8s %6s  (calls, calls, calls, iterations)\n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
    int ns[] = {10, 15, 20, 25, 30};
    for (int n : ns)
    {
        long long f = FibIt(n);
        fibCalls = 0;
        long long r1 = FibRec(n);
        long long c1 = fibCalls;
        fibCalls = 0;
        vector<long long> memo(n + 1, -1);
        long long r2 = FibMemo(n, memo.data());
        long long c2 = fibCalls;
        bool ok = (r1 == r2 && r2 == f && c1 == 2 * FibIt(n + 1) - 1 && c2 == 2 * n - 1);
        printf("%3d %8lld %9lld %10lld %8lld %6d  %s\n", n, f, c1, 2 * FibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (int n : {20, 25, 30})
    {
        double t0 = NowMs(); volatile long long sink = FibRec(n); double t1 = NowMs() - t0;
        t0 = NowMs();
        for (int rep = 0; rep < 1000; rep++)
        {
            sink = FibIt(n);
        }
        double t2 = (NowMs() - t0) / 1000;
        printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms\n", n, t1, t2);
    }
    return 0;
}
#endif

Writing fibonacci.cpp


In [18]:
!g++ -std=c++17 -O2 -o fibonacci fibonacci.cpp && ./fibonacci

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     0.0074 ms   FibIt   0.000004 ms
n=25  FibRec     0.0947 ms   FibIt   0.000004 ms
n=30  FibRec     0.9610 ms   FibIt   0.000007 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [19]:
%%writefile prova_fibonacci.cpp
#define NOMAIN
#include "fibonacci.cpp"

int main()
{
    // Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
    int n = 32;
    fibCalls = 0;
    long long r1 = FibRec(n);
    printf("%lld %lld\n", r1, fibCalls);
    fibCalls = 0;
    vector<long long> memo(n + 1, -1);
    long long r2 = FibMemo(n, memo.data());
    printf("%lld %lld\n", r2, fibCalls);
    return 0;
}

Writing prova_fibonacci.cpp


In [20]:
!g++ -std=c++17 -O2 -o prova_fibonacci prova_fibonacci.cpp && ./prova_fibonacci

2178309 7049155
2178309 63


## 6. Tail recursion and its elimination: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

A recursive call is a *tail call* if it is the last operation and its result is returned as it is: it can be replaced by an update of the parameters and a jump to the beginning, that is, by a loop (loop condition = negation of the base case; tail call = simultaneous assignment of the parameters). The result is the same, the number of steps has the same order of growth and the extra memory goes down from $\Theta(\text{depth})$ to $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: two natural numbers $a$, $b$. *Output*: their greatest common divisor (Euclid's algorithm). With $(a,b)=(252,105)$ both go through the pairs $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ and return $21$: four calls against three iterations. The temporary variable $r$ is needed because $a\bmod b$ must be computed with the *old* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: a sorted array, two indices $l$, $r$ and a value $m$. *Output*: an index of $m$ in $a[l..r]$ or $-1$. `ItBS(a, 1, n, m)` is the iterative binary search of the chapter "Search".

In [21]:
%%writefile tail_recursion.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

vector<string> gcdLog;   // pairs (a, b) visited (only to print them)
int rbsCalls = 0;        // calls of RecBS
int bsIters = 0;         // iterations of ItBS

string PairStr(long long a, long long b)
{
    return "(" + to_string(a) + "," + to_string(b) + ")";
}

// GCDRec(a, b)
long long GCDRec(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    if (b == 0)
    {
        return a;
    }
    return GCDRec(b, a % b);            // tail call
}

// GCDIt(a, b)
long long GCDIt(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    while (b != 0)                      // negation of the base case
    {
        long long r = a % b;            // with the old a
        a = b;
        b = r;
        gcdLog.push_back(PairStr(a, b));
    }
    return a;
}

// RecBS(a, l, r, m)
int RecBS(int *a, int l, int r, int m)
{
    rbsCalls++;
    if (l > r)
    {
        return -1;
    }
    int mid = l + (r - l) / 2;
    if (a[mid] == m)
    {
        return mid;
    }
    if (a[mid] < m)
    {
        return RecBS(a, mid + 1, r, m);     // tail call
    }
    else
    {
        return RecBS(a, l, mid - 1, m);     // tail call
    }
}

// ItBS(a, l, r, m)
int ItBS(int *a, int l, int r, int m)
{
    while (l <= r)                          // negation of the base case l > r
    {
        bsIters++;
        int mid = l + (r - l) / 2;
        if (a[mid] == m)
        {
            return mid;                     // the second base case stays in the loop
        }
        if (a[mid] < m)
        {
            l = mid + 1;
        }
        else
        {
            r = mid - 1;
        }
    }
    return -1;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

#ifndef NOMAIN
int main()
{
    long long pairsAB[][2] = {{252, 105}, {1071, 462}, {832040, 514229}};
    for (auto &p : pairsAB)
    {
        long long a = p[0], b = p[1];
        gcdLog.clear();
        long long g1 = GCDRec(a, b);
        int calls = gcdLog.size();
        string pairs;
        for (size_t i = 0; i < gcdLog.size(); i++)
        {
            if (i > 0) pairs += " ";
            pairs += gcdLog[i];
        }
        gcdLog.clear();
        long long g2 = GCDIt(a, b);
        int iters = (int)gcdLog.size() - 1;
        printf("GCDRec(%lld, %lld) = %lld   calls=%d     GCDIt = %lld   iterations=%d   %s\n", a, b, g1, calls, g2, iters, g1 == g2 ? "OK" : "ERROR");
        if (a == 252)
        {
            printf("  pairs: %s\n", pairs.c_str());
        }
    }

    printf("\n");
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("%4s %6s %6s %6s %11s\n", "m", "RecBS", "calls", "ItBS", "iterations");
    int ms[] = {12, 10, 3, 19, 100};
    for (int m : ms)
    {
        rbsCalls = 0;
        int r1 = RecBS(a, 0, n - 1, m);
        bsIters = 0;
        int r2 = ItBS(a, 0, n - 1, m);
        printf("%4d %6d %6d %6d %11d   %s\n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
    }
    return 0;
}
#endif

Writing tail_recursion.cpp


In [22]:
!g++ -std=c++17 -O2 -o tail_recursion tail_recursion.cpp && ./tail_recursion

GCDRec(252, 105) = 21   calls=4     GCDIt = 21   iterations=3   OK
  pairs: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   calls=4     GCDIt = 21   iterations=3   OK
GCDRec(832040, 514229) = 1   calls=29     GCDIt = 1   iterations=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Try it yourself.** Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.

In [23]:
%%writefile prova_tail_recursion.cpp
#define NOMAIN
#include "tail_recursion.cpp"

int main()
{
    // Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.
    gcdLog.clear();
    long long g = GCDRec(46368, 28657);
    printf("%lld %d\n", g, (int)gcdLog.size());
    return 0;
}

Writing prova_tail_recursion.cpp


In [24]:
!g++ -std=c++17 -O2 -o prova_tail_recursion prova_tail_recursion.cpp && ./prova_tail_recursion

1 23


## 7. Depth of the recursion and the call stack

The memory used by the call stack is proportional to the **depth** $D(n)$ of the recursion (the maximum number of calls active at the same time), not to the number of calls. The table compares the **measured** depth with the formula of the chapter:

| algorithm | $D(n)$ |
|---|---|
| sum by decrease by one (base case $n=0$) | $n+1$ |
| sum by halving | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (worst case, absent value) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

A depth $\Theta(\log_2 n)$ is never a problem in practice; a depth $\Theta(n)$ risks a **stack overflow**: the last part of the cell (the lines starting with `~~`) depends on the language and on the machine.

In [25]:
%%writefile depth.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

#include <sys/resource.h>

int depth = 0;
int maxDepth = 0;
int calls = 0;

void Enter()
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
}

long long SumRecDecrease(int *a, int n)
{
    Enter();
    long long result = (n == 0) ? 0 : SumRecDecrease(a, n - 1) + a[n - 1];
    depth--;
    return result;
}

long long SumRecHalving(int *a, int l, int r)
{
    Enter();
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = SumRecHalving(a, l, mid) + SumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long FastPower(long long x, int n)
{
    Enter();
    long long result;
    if (n == 0)
    {
        result = 1;
    }
    else
    {
        long long y = FastPower(x, n / 2);
        result = (n % 2 == 0) ? y * y : x * y * y;
    }
    depth--;
    return result;
}

int RecBS(int *a, int l, int r, int m)
{
    calls++;
    if (l > r)
    {
        return -1;
    }
    int mid = l + (r - l) / 2;
    if (a[mid] == m)
    {
        return mid;
    }
    if (a[mid] < m)
    {
        return RecBS(a, mid + 1, r, m);
    }
    return RecBS(a, l, mid - 1, m);
}

long long FibRec(int n)
{
    Enter();
    long long result = (n <= 1) ? n : FibRec(n - 1) + FibRec(n - 2);
    depth--;
    return result;
}

int FloorLog2(int n)
{
    int f = 0;
    while ((1 << (f + 1)) <= n) f++;
    return f;
}

int CeilLog2(int n)
{
    int c = 0;
    while ((1 << c) < n) c++;
    return c;
}

#ifndef NOMAIN
int main()
{
    printf("%5s %9s %5s %8s %8s %10s %8s %6s %6s\n", "n", "decrease", "n+1", "halving", "formula", "FastPower", "formula", "RecBS", "bound");
    int ns[] = {10, 100, 500};
    for (int n : ns)
    {
        vector<int> a(n);
        for (int i = 0; i < n; i++) a[i] = i;
        maxDepth = 0; SumRecDecrease(a.data(), n); int d1 = maxDepth;
        maxDepth = 0; SumRecHalving(a.data(), 0, n - 1); int d2 = maxDepth;
        maxDepth = 0; FastPower(1, n); int d3 = maxDepth;
        calls = 0; RecBS(a.data(), 0, n - 1, n + 100); int d4 = calls;
        printf("%5d %9d %5d %8d %8d %10d %8d %6d %6d\n", n, d1, n + 1, d2, CeilLog2(n) + 1, d3, FloorLog2(n) + 2, d4, FloorLog2(n) + 2);
    }
    printf("\n");
    printf("%3s %13s\n", "n", "FibRec depth");
    int fs[] = {5, 10, 20};
    for (int n : fs)
    {
        maxDepth = 0;
        FibRec(n);
        printf("%3d %13d\n", n, maxDepth);
    }

    struct rlimit rl;
    getrlimit(RLIMIT_STACK, &rl);
    printf("~~ C++: stack limit = %lld KB; a frame of 32-48 bytes allows a depth of about %lld calls\n", (long long)rl.rlim_cur / 1024, (long long)rl.rlim_cur / 40);
    printf("~~ C++: sumRecDecrease with n = 10^7 would need about 320 MB of stack: segmentation fault (not run here)\n");
    return 0;
}
#endif

Writing depth.cpp


In [26]:
!g++ -std=c++17 -O2 -o depth depth.cpp && ./depth

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ C++: stack limit = 8192 KB; a frame of 32-48 bytes allows a depth of about 209715 calls
~~ C++: sumRecDecrease with n = 10^7 would need about 320 MB of stack: segmentation fault (not run here)


**Try it yourself.** Measure the depth for another $n$ (here $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [27]:
%%writefile prova_depth.cpp
#define NOMAIN
#include "depth.cpp"

int main()
{
    // Measure the depth of the recursion for another n (here 2000): decrease = n+1, halving = ceil(log2 n) + 1.
    int n = 2000;
    vector<int> a(n, 1);
    maxDepth = 0;
    SumRecDecrease(a.data(), n);
    printf("%d ", maxDepth);
    maxDepth = 0;
    SumRecHalving(a.data(), 0, n - 1);
    printf("%d\n", maxDepth);
    return 0;
}

Writing prova_depth.cpp


In [28]:
!g++ -std=c++17 -O2 -o prova_depth prova_depth.cpp && ./prova_depth

2001 12
